# Foundation Model GPU Demo — Discrete-Time Hazard Adapters

This notebook demonstrates **TabPFN** and **TabICL** pooled discrete-time hazard foundation adapters on SurvArena's built-in survival datasets, with explicit **GPU verification** for Google Colab.

**What it covers**
- Colab GPU runtime detection (`cuda` availability, device name, `nvidia-smi`)
- Install SurvArena with foundation extras (`tabpfn`, `tabicl`)
- Run `tabpfn_discrete_hazard_survival` and `tabicl_discrete_hazard_survival` across all built-in clinical datasets
- Compare Uno C-index and runtime between methods
- Optional section for TCGA genomics datasets (requires artifact preparation)

**Colab setup (required for GPU)**
1. Open this notebook in Google Colab.
2. **Runtime → Change runtime type → Hardware accelerator → GPU** (T4/L4/A100).
3. Run cells top-to-bottom.

> Foundation adapters delegate device selection to TabPFN/TabICL. On Colab Linux + CUDA, defaults are `device: auto` (TabPFN → `cuda:0`) and `device: null` (TabICL → `cuda` when available).

In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Running in Colab: {IN_COLAB}")

if IN_COLAB:
    print("\nColab detected. Ensure Runtime → Change runtime type → GPU is selected.")
else:
    print("\nLocal/non-Colab environment. CUDA will be used only if a CUDA GPU is available.")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_index)
    print(f"CUDA device: {props.name}")
    print(f"CUDA capability: {props.major}.{props.minor}")
    print(f"Total GPU memory (GiB): {props.total_memory / (1024 ** 3):.2f}")
    # Lightweight tensor op to confirm the GPU is usable
    probe = torch.ones(1024, device="cuda")
    _ = probe @ probe.T
    print("GPU tensor probe: OK")
else:
    print("WARNING: CUDA is not available. Foundation models will fall back to CPU.")
    if IN_COLAB:
        print("→ In Colab: Runtime → Change runtime type → GPU, then re-run from the top.")

# Optional: nvidia-smi snapshot (Colab/Linux with NVIDIA driver)
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True, check=True, timeout=10)
    print("\nnvidia-smi:")
    print(smi.stdout.strip())
except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
    print(f"\nnvidia-smi not available ({type(exc).__name__})")

## 1. Install SurvArena with foundation extras

Clone the repository (Colab) or use the local checkout, then install TabPFN + TabICL extras.

In [ ]:
SURVARENA_REPO = os.environ.get("SURVARENA_REPO", "https://github.com/MasalaKimchi/SurvArena.git")
REPO_DIR = Path(os.environ.get("SURVARENA_DIR", "/content/SurvArena" if IN_COLAB else Path.cwd().parent))

if IN_COLAB and not (REPO_DIR / "pyproject.toml").exists():
    !git clone --depth 1 {SURVARENA_REPO} {REPO_DIR}

if not (REPO_DIR / "pyproject.toml").exists():
    # Fallback: notebook lives in demo/ inside the repo
    candidate = Path.cwd()
    while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
        candidate = candidate.parent
    if (candidate / "pyproject.toml").exists():
        REPO_DIR = candidate

assert (REPO_DIR / "pyproject.toml").exists(), f"Could not locate SurvArena repo at {REPO_DIR}"
print(f"Repo root: {REPO_DIR}")

%cd {REPO_DIR}
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q -e ".[foundation-tabpfn,foundation-tabarena]"

### Hugging Face token (TabPFN gated weights)

TabPFN v2.5 weights are gated on Hugging Face. Set a token before running if weights are not cached.

In Colab you can use **Secrets** (`HF_TOKEN`) or run `hf auth login` interactively.

In [ ]:
# Colab: store HF_TOKEN in notebook secrets, or export manually before running.
if IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN loaded from Colab secrets.")
    except Exception:
        if os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN"):
            print("HF token found in environment.")
        else:
            print("No HF token configured. TabPFN may fail on first download of gated weights.")
else:
    print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")))

In [ ]:
from survarena.methods.foundation.readiness import foundation_runtime_catalog

for status in foundation_runtime_catalog():
    if status.method_id in {"tabpfn_discrete_hazard_survival", "tabicl_discrete_hazard_survival"}:
        print(f"{status.method_id}: ready={status.runtime_ready}")
        if status.blocked_reason:
            print(f"  blocked: {status.blocked_reason}")
        if status.warning_reason:
            print(f"  warning: {status.warning_reason}")

## 2. Verify foundation adapters resolve to GPU

Quick synthetic fit to confirm TabPFN and TabICL place model weights on CUDA before the full benchmark.

In [ ]:
import numpy as np
from survarena.methods.registry import get_method_class


def _resolved_device_label(model) -> str:
    for attr in ("device_", "device"):
        value = getattr(model, attr, None)
        if value is not None:
            return str(value)
    for attr in ("models_", "model_"):
        container = getattr(model, attr, None)
        if container is None:
            continue
        items = container if isinstance(container, list) else [container]
        for item in items:
            if item is None:
                continue
            for inner_attr in ("device_", "device"):
                inner = getattr(item, inner_attr, None)
                if inner is not None:
                    return str(inner)
    return "unknown"


def probe_foundation_device(method_id: str, *, force_cuda: bool = True) -> str:
    cls = get_method_class(method_id)
    params: dict = {"n_estimators": 1, "n_intervals": 3, "min_rows_per_interval": 5, "max_stacked_rows": 2000, "seed": 0}
    if method_id.startswith("tabpfn"):
        params["device"] = "cuda" if force_cuda and torch.cuda.is_available() else "cpu"
    elif method_id.startswith("tabicl"):
        params["device"] = "cuda" if force_cuda and torch.cuda.is_available() else "cpu"
    method = cls(**params)

    rng = np.random.default_rng(0)
    n = 80
    X = rng.normal(size=(n, 6)).astype(np.float32)
    time = rng.exponential(2.0, size=n)
    event = rng.integers(0, 2, size=n)

    method.fit(X, time, event)
    return _resolved_device_label(method)


for mid in ["tabpfn_discrete_hazard_survival", "tabicl_discrete_hazard_survival"]:
    device_label = probe_foundation_device(mid)
    on_cuda = "cuda" in device_label.lower()
    print(f"{mid}: resolved device = {device_label} | GPU active = {on_cuda}")
    if torch.cuda.is_available() and not on_cuda:
        print("  WARNING: CUDA is available but adapter did not resolve to a CUDA device.")

## 3. Configure and run discrete-hazard benchmark

Runs across all **7 built-in clinical datasets** with the manuscript profile contract (`outer_folds=3`, `outer_repeats=3`). The run cell uses `limit_seeds=1` for a faster Colab demo (3 folds × 1 repeat per dataset/method). Remove `limit_seeds` for the full 3-repeat evaluation.

In [ ]:
from copy import deepcopy

# All built-in clinical datasets (no external TCGA artifacts required)
CLINICAL_DATASETS = [
    "support",
    "metabric",
    "nwtco",
    "aids",
    "gbsg2",
    "flchain",
    "whas500",
]

FOUNDATION_METHODS = [
    "tabpfn_discrete_hazard_survival",
    "tabicl_discrete_hazard_survival",
]

DEMO_BENCHMARK_CFG = {
    "benchmark_id": "demo_foundation_discrete_hazard_gpu",
    "task_type": "right_censored_survival",
    "split_strategy": "repeated_nested_cv",
    # Manuscript profile requires outer_folds>=3 and outer_repeats>=3.
    # Use limit_seeds=1 in the run cell for a faster Colab demo (3 folds x 1 repeat).
    "outer_folds": 3,
    "outer_repeats": 3,
    "inner_folds": 2,
    "seeds": [11, 22, 33],
    "primary_metric": "uno_c",
    "profile": "manuscript",
    "comparison_modes": ["no_hpo"],
    "secondary_metrics": ["harrell_c", "ibs", "td_auc", "brier"],
    "autogluon": {
        "enabled": False,
        "presets": None,
        "time_limit_seconds": 60,
        "hyperparameter_tune_kwargs": None,
        "num_bag_folds": 0,
        "num_stack_levels": 0,
        "refit_full": False,
    },
    "hpo": {
        "enabled": False,
        "max_trials": 0,
        "timeout_seconds": 60,
        "sampler": "random",
        "pruner": "none",
        "n_startup_trials": 0,
        "method_overrides": {
            "tabpfn_discrete_hazard_survival": {
                "enabled": False,
                "search_space": None,
                "default_params": {
                    "n_estimators": 1,
                    "n_intervals": 5,
                    "min_rows_per_interval": 20,
                    "max_stacked_rows": 50000,
                    "device": "cuda" if torch.cuda.is_available() else "cpu",
                },
            },
            "tabicl_discrete_hazard_survival": {
                "enabled": False,
                "search_space": None,
                "default_params": {
                    "n_estimators": 1,
                    "n_intervals": 5,
                    "min_rows_per_interval": 20,
                    "max_stacked_rows": 50000,
                    "device": "cuda" if torch.cuda.is_available() else "cpu",
                    "use_amp": torch.cuda.is_available(),
                },
            },
        },
    },
    "time_horizons_quantiles": [0.25, 0.5, 0.75],
    "exports": {"profile": "core_csv", "manuscript_artifact_layout": "compact"},
    "datasets": CLINICAL_DATASETS,
    "methods": FOUNDATION_METHODS,
}

OUTPUT_DIR = REPO_DIR / "results" / "demo" / "foundation_discrete_hazard_gpu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from survarena.benchmark.overview import benchmark_plan

plan = benchmark_plan(REPO_DIR, DEMO_BENCHMARK_CFG)
print("Planned run units:", plan["planned_run_units"])
print("Datasets:", plan["datasets"])
print("Methods:", plan["methods"])
print("Output:", OUTPUT_DIR)

In [ ]:
from survarena.benchmark.runner import run_benchmark

benchmark_cfg = deepcopy(DEMO_BENCHMARK_CFG)

# Validate wiring without fitting models
run_benchmark(
    repo_root=REPO_DIR,
    benchmark_cfg=benchmark_cfg,
    limit_seeds=1,
    dry_run=True,
    output_dir=OUTPUT_DIR,
)

print("Dry run OK. Starting benchmark...")
if torch.cuda.is_available():
    print(f"Expecting GPU execution on: {torch.cuda.get_device_name(0)}")

run_benchmark(
    repo_root=REPO_DIR,
    benchmark_cfg=benchmark_cfg,
    limit_seeds=1,
    dry_run=False,
    output_dir=OUTPUT_DIR,
    resume=True,
    max_retries=0,
    execution_n_jobs_override=1,
)

print("Benchmark complete.")

## 4. Compare TabPFN vs TabICL results

In [ ]:
import pandas as pd
from IPython.display import display
from survarena.benchmark.overview import benchmark_report

report = benchmark_report(OUTPUT_DIR)
print("Status counts:", report["status_counts"])
print("Primary metric:", report.get("primary_metric"))

fold_paths = sorted(OUTPUT_DIR.rglob("*_fold_results.csv"))
fold_results = pd.concat([pd.read_csv(p) for p in fold_paths], ignore_index=True)

success = fold_results[fold_results["status"].astype(str).eq("success")].copy()
print(f"\nSuccessful folds: {len(success)} / {len(fold_results)}")

primary_metric = report.get("primary_metric", "uno_c")
display_cols = ["dataset_id", "method_id", "outer_fold", "seed", primary_metric, "harrell_c", "runtime_sec", "status"]
display(success.sort_values(["dataset_id", "method_id", "outer_fold"])[display_cols].head(20))

In [ ]:
# Dataset-level means for head-to-head comparison
metric_cols = [c for c in [primary_metric, "harrell_c", "ibs", "runtime_sec"] if c in success.columns]
summary = (
    success.groupby(["dataset_id", "method_id"], as_index=False)[metric_cols]
    .mean(numeric_only=True)
    .sort_values(["dataset_id", primary_metric], ascending=[True, False])
)

pivot = summary.pivot(index="dataset_id", columns="method_id", values=primary_metric)
pivot["delta_tabpfn_minus_tabicl"] = (
    pivot.get("tabpfn_discrete_hazard_survival") - pivot.get("tabicl_discrete_hazard_survival")
)

print(f"Mean {primary_metric} by dataset and method:")
display(summary)

print("\nHead-to-head (TabPFN − TabICL):")
display(pivot.sort_values("delta_tabpfn_minus_tabicl", ascending=False))

wins = (pivot["delta_tabpfn_minus_tabicl"] > 0).sum()
losses = (pivot["delta_tabpfn_minus_tabicl"] < 0).sum()
ties = (pivot["delta_tabpfn_minus_tabicl"] == 0).sum()
print(f"\nTabPFN wins / loses / ties on {primary_metric}: {wins} / {losses} / {ties} datasets")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if success.empty:
    print("No successful folds to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.barplot(
        data=summary,
        x="dataset_id",
        y=primary_metric,
        hue="method_id",
        ax=axes[0],
    )
    axes[0].set_title(f"Mean {primary_metric} by dataset")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].legend(title="method", fontsize=8)

    if "runtime_sec" in summary.columns:
        sns.barplot(
            data=summary,
            x="dataset_id",
            y="runtime_sec",
            hue="method_id",
            ax=axes[1],
        )
        axes[1].set_title("Mean runtime (sec) by dataset")
        axes[1].tick_params(axis="x", rotation=45)
        axes[1].legend(title="method", fontsize=8)

    plt.tight_layout()
    plt.show()

    if torch.cuda.is_available():
        try:
            alloc_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            reserved_mb = torch.cuda.max_memory_reserved() / (1024 ** 2)
            print(f"Peak CUDA memory — allocated: {alloc_mb:.1f} MiB, reserved: {reserved_mb:.1f} MiB")
        except Exception as exc:
            print(f"Could not read CUDA memory stats: {exc}")

## 5. (Optional) TCGA genomics datasets

The five TCGA Xena cohorts require prepared parquet artifacts. On Colab, download and prepare them before adding to the benchmark config.

```bash
python scripts/prepare_cancer_survival_datasets.py --dataset all --max-genes 1000
```

Then extend `DEMO_BENCHMARK_CFG["datasets"]` with:

- `tcga_brca_xena_1000`
- `tcga_luad_xena_1000`
- `tcga_kirc_xena_1000`
- `tcga_skcm_xena_1000`
- `tcga_ov_xena_1000`

> Genomics runs are memory-intensive. A Colab T4 with 1000 genes may require smaller `predict_batch_size` or `n_estimators: 1`.

In [ ]:
# Uncomment to prepare TCGA artifacts on Colab (long-running download + preprocessing).
# !{sys.executable} scripts/prepare_cancer_survival_datasets.py --dataset all --max-genes 1000

TCGA_DATASETS = [
    "tcga_brca_xena_1000",
    "tcga_luad_xena_1000",
    "tcga_kirc_xena_1000",
    "tcga_skcm_xena_1000",
    "tcga_ov_xena_1000",
]

def _tcga_parquet_path(dataset_id: str) -> Path:
    return REPO_DIR / "data" / "processed" / "cancer_survival_1000" / f"{dataset_id.replace('_1000', '')}.parquet"

available_tcga = [d for d in TCGA_DATASETS if (REPO_DIR / "configs" / "datasets" / f"{d}.yaml").exists()]
prepared_tcga = [d for d in TCGA_DATASETS if _tcga_parquet_path(d).exists()]
print("TCGA dataset configs found:", available_tcga)
print("Prepared parquet present for:", prepared_tcga)